# Chapter 4 — Embeddings and the Residual Stream

Every number in Chapter 4, recomputed. Each cell **asserts** what it checks, so
`python tests/run_notebooks.py 04` fails if the chapter and the code disagree. GPT-2's weights
would need a download, so Exercise 4.2 (nearest neighbours) and the logit lens are not reproduced
here; Exercise 4.3 runs on this course's model (`tfs/model.py`) trained for a few seconds on the
bundled offline corpus.

In [ ]:
import math, sys, os
sys.path.insert(0, os.path.abspath(".."))      # run from notebooks/
sys.path.insert(0, os.path.abspath("."))       # or from the repo root
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)
from tfs.model import Config, Model

## 4.1 — an embedding is a lookup: shapes and the GPT-2 table

In [ ]:
V, C = 50257, 768
emb = nn.Embedding(V, C)
print(tuple(emb.weight.shape), f"{emb.weight.numel():,} parameters")
assert emb.weight.numel() == 38_597_376 and round(emb.weight.numel() / 1e6, 1) == 38.6
ids = torch.tensor([[15496, 995, 0], [464, 3797, 3332]])          # (B=2, T=3)
x = emb(ids)
assert ids.shape == (2, 3) and x.shape == (2, 3, 768)              # (B, T) -> (B, T, C)
onehot = torch.zeros(1, V); onehot[0, 15496] = 1.0
assert torch.allclose(onehot @ emb.weight, emb.weight[15496][None])  # identical answers

## 4.1 — the worked example: the same row, two ways, and the waste

In [ ]:
W = torch.arange(24, dtype=torch.float64).reshape(6, 4) / 10        # id i -> [0.4i, ..., 0.4i+0.3]
assert W[4].tolist() == [1.6, 1.7, 1.8, 1.9] and W[5].tolist() == [2.0, 2.1, 2.2, 2.3]
oh = torch.zeros(6, dtype=torch.float64); oh[3] = 1
print("matmul", (oh @ W).tolist(), " lookup", W[3].tolist())
assert torch.allclose(oh @ W, torch.tensor([1.2, 1.3, 1.4, 1.5], dtype=torch.float64))
assert torch.equal(oh @ W, W[3])
mults, zero = 6 * 4, 5 * 4
print(f"{zero} of {mults} multiplies by zero = {zero / mults:.0%}")
assert round(100 * zero / mults) == 83
gpt2 = 50257 * 768
print(f"GPT-2: matmul {gpt2:,} multiplies, lookup 768 reads -> {gpt2 // 768:,}x the arithmetic")
assert gpt2 == 38_597_376 and gpt2 // 768 == 50257 and 50257 - 1 == 50256

## 4.1 — the gradient reaches exactly one row per token, and what AdamW does afterwards

A token that appears once gets one gradient. Its row then keeps moving: on momentum, and from
weight decay, which shrinks it every step.

**Resolved (November 2026).** This cell found an error in the chapter, and the chapter text was corrected to match the measurement below. The original finding: The chapter says the row "keeps drifting for a few hundred steps on
momentum". After the gradient stops, Adam's step is about m̂/√v̂, which decays like
(β₁/√β₂)ᵗ: under 1% of its first size after ~45 steps with PyTorch's default betas, and the
measurement below agrees. Momentum drift is over within tens of steps; what keeps moving the row
for hundreds of steps is weight decay (lr·λ·w every step).

In [ ]:
def rare_token_moves(weight_decay, betas, steps=400):
    torch.manual_seed(0)
    e = nn.Embedding(10, 4)
    opt = torch.optim.AdamW(e.parameters(), lr=1e-2, weight_decay=weight_decay, betas=betas)
    moves = []
    for step in range(steps):
        ids = torch.tensor([3]) if step == 0 else torch.tensor([1])   # token 3 appears once, ever
        before = e.weight[3].detach().clone()
        opt.zero_grad()
        (e(ids) * torch.randn(1, 4)).sum().backward()
        if step == 0:
            rows = (e.weight.grad.abs().sum(1) > 0).nonzero().flatten().tolist()
            assert rows == [3]                                     # exactly one row has a gradient
        opt.step()
        moves.append((e.weight[3].detach() - before).norm().item())
    return moves

for betas in ((0.9, 0.999), (0.9, 0.95)):
    m = rare_token_moves(0.0, betas)
    gone = next(t for t, v in enumerate(m) if v < 0.01 * m[0])
    print(f"betas {betas}: momentum drift falls below 1% of its first step after {gone} steps")
    assert gone < 100                                              # tens of steps, not hundreds
m = rare_token_moves(0.1, (0.9, 0.95))
print(f"with weight decay 0.1: still moving {m[300]:.1e} per step at step 300")
assert m[300] > 1e-4                                               # decay keeps shrinking the row

## 4.2 — weight tying: V·C parameters instead of 2·V·C

In [ ]:
cfg = Config(vocab_size=1000, d_model=64, n_layer=2, n_head=4, n_kv_head=2, d_head=16, d_ff=128, max_T=64)
tied = Model(cfg)
untied = Model(Config(**{**cfg.__dict__, "tie_embed": False}))
n = lambda m: sum(p.numel() for p in m.parameters())
print("tied", n(tied), " untied", n(untied))
assert tied.head.weight is tied.embed.weight and tied.head.weight.shape == (1000, 64)  # both (V, C)
assert n(untied) - n(tied) == 1000 * 64 and n(tied) == cfg.n_params()

## 4.3 — the residual stream has one shape on every line

In [ ]:
shapes = []
hooks = [b.register_forward_hook(lambda mod, inp, out: shapes.append(tuple(out[0].shape)))
         for b in tied.blocks]
tied(torch.randint(0, 1000, (3, 17)))
for h in hooks: h.remove()
print(shapes)
assert shapes == [(3, 17, 64)] * cfg.n_layer                      # (B, T, C) after every block

## 4.4 — why x + f(x): 0.9⁸⁰, and the 1 + f′(x) term

In [ ]:
print(f"0.9**80 = {0.9 ** 80:.2e}")
assert round(0.9 ** 80, 4) == 0.0002
x = torch.tensor(2.0, requires_grad=True)
(x + torch.tanh(0.5 * x)).backward()
fprime = 0.5 * (1 - torch.tanh(torch.tensor(1.0)) ** 2)
assert torch.isclose(x.grad, 1 + fprime)                           # d(x + f)/dx = 1 + f'(x)

## Quiz — how much of an 8B model is embedding tables?

In [ ]:
table = 128256 * 4096
print(f"one table {table:,}; two {2 * table / 1e9:.2f} B = {2 * table / 8e9:.0%} of 8 B")
assert round(table / 1e6) == 525 and round(2 * table / 1e9, 2) == 1.05
assert round(100 * 2 * table / 8e9) == 13 and 2 * table / 1e9 > 1.0   # over 100% of a 1B model

## Exercise 4.1 — embedding = one-hot matmul, and the speed ratio

The chapter predicts the lookup is typically 100–1000× faster at V = 50,000. Timings depend on
the machine and its load: this notebook measured 120–1000× on a busy 8-core CPU. The assert is a
deliberately loose 30× so that a loaded machine does not fail it; the printed ratio is the result.

In [ ]:
import time
e10 = nn.Embedding(10, 4)
assert torch.equal(e10(torch.tensor([3])), F.one_hot(torch.tensor([3]), 10).float() @ e10.weight)
E = nn.Embedding(50000, 768)
ids = torch.randint(0, 50000, (8, 32))
def bench(fn, reps=5):
    fn(); t = time.perf_counter()
    for _ in range(reps): fn()
    return (time.perf_counter() - t) / reps
with torch.no_grad():
    assert torch.allclose(E(ids), F.one_hot(ids, 50000).float() @ E.weight)
    t_lookup = bench(lambda: E(ids))
    t_matmul = bench(lambda: F.one_hot(ids, 50000).float() @ E.weight)
print(f"lookup {t_lookup * 1e3:.3f} ms   one-hot matmul {t_matmul * 1e3:.1f} ms   ratio {t_matmul / t_lookup:,.0f}x")
assert t_matmul / t_lookup > 30

## Exercise 4.3 — break the residual stream

The same model with `x = attn(norm(x))` and `x = mlp(norm(x))` in every block (residual removed),
trained side by side on the bundled offline corpus. The no-residual model trains dramatically
worse, as the chapter says.

**Resolved (November 2026).** This cell found an error in the chapter, and the chapter text was corrected to match the measurement below. The original finding: Two of the exercise's predictions do not hold for this pre-norm (RMSNorm) model:
(1) the gap is not "marginal at 2 layers" — it is already large at 2 layers; (2) the no-residual
model's first-layer gradients are *not* orders of magnitude smaller — they are about as large or
larger. With a norm in front of every sublayer, each block rescales its input, so the gradient
does not vanish; what is lost is the identity path that carries each token's own embedding
forward. (The vanishing-gradient picture holds for an unnormalised stack — see the Chapter 10
notebook, Exercise 10.3.)

In [ ]:
from tfs.train import _fallback_corpus, get_batch
from tfs.tokenizer import CharTokenizer
text = _fallback_corpus()[:60000]
ctok = CharTokenizer(text)
data = torch.tensor(ctok.encode(text))

def no_residual(self, x, cos, sin, cache=None):
    a, c = self.attn(self.n1(x), cos, sin, cache)
    x = a                                       # was: x = x + a
    x = self.mlp(self.n2(x))                    # was: x = x + self.mlp(self.n2(x))
    return x, c

def run(n_layer, residual, steps=60):
    torch.manual_seed(0)
    m = Model(Config(vocab_size=ctok.vocab_size, d_model=48, n_layer=n_layer, n_head=4,
                     n_kv_head=4, d_head=12, d_ff=128, max_T=64))
    if not residual:
        for b in m.blocks:
            b.forward = no_residual.__get__(b)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    torch.manual_seed(1)
    for step in range(steps):
        x = get_batch(data, 16, 64)
        _, loss, _ = m(x, targets=x)
        opt.zero_grad(); loss.backward()
        if step == 0:
            g0 = m.blocks[0].attn.q_proj.weight.grad.norm().item()
        opt.step()
    return loss.item(), g0

res = {(L, r): run(L, r) for L in (2, 4) for r in (True, False)}
for (L, r), (loss, g0) in res.items():
    print(f"{L} layers  residual={r!s:5}  loss after 60 steps {loss:.3f}   block[0] grad {g0:.2e}")
for L in (2, 4):
    assert res[L, False][0] > res[L, True][0] + 0.5            # dramatically worse, even at 2 layers
    assert res[L, False][1] > 0.1 * res[L, True][1]            # first-layer grads NOT orders smaller